# 3. Patient samples: a CSV table and an AnnData

A sample query is a whole expression profile on any scale; it is converted to within-sample ranks. Give the rest of
the cohort as the **background** so that the match reflects what is specific to the patient. Tumour-vs-cell-line
differences shared by all samples are then centred away.
- **Table (CSV/TSV):** genes × samples; pick one with `sample=`. The other columns become the background.
- **AnnData:** pick an observation with `sample=`, or pseudobulk a group with `groupby=` / `group=`. Counts are summed and log-CPM normalised; log data are averaged.

The demo data are 45 BeatAML diagnosis bone-marrow samples: 15 KMT2A-r, 15 NPM1-mutant and 15 CBFB-MYH11 (BeatAML 2.0, Bottomly et al. 2022 *Cancer Cell* 40:850–864, doi:10.1016/j.ccell.2022.07.002). They are downloaded the first time with `al.datasets.download("demo")`, which runs automatically below.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import pandas as pd
import anchorline as al

pd.set_option("display.width", 200); pd.set_option("display.max_columns", 20); pd.set_option("display.max_colwidth", 60)
al.settings.figdir = "figures"          # save=True writes here
al.settings.set_figure_params(fontsize=8.5, dpi=150)
atlas = al.load()                       # first time: downloads the atlas (~560 MB) to al.settings.data_dir
atlas

In [ ]:
D = al.datasets.demo_path().parent          # downloads the demo set the first time
samples = pd.read_csv(D / "beataml_demo_samples.csv", index_col=0)
samples.group.value_counts()

## A sample from a CSV table

In [ ]:
sid = samples.index[samples.group == "KMT2A-r"][0]
print(sid, samples.loc[sid, "fusion"])
mt = al.tl.match(atlas, D / "beataml_demo_expression.csv", sample=sid)
mt

In [ ]:
al.pl.match_ranking(mt, n_top=12)
al.pl.pseudotime(mt, n_top=5)            # the star = this patient's position

In [ ]:
dr = al.tl.rank_drugs(atlas, mt, cell_lines=5)
al.pl.drug_ranking(dr, n_top=12)
al.pl.drug_shifts(dr, n_top=10)
al.pl.state_map(dr, n_top=6)

In [ ]:
al.pl.gene_changes(dr, genes=["CLEC12A", "CD84"], n_top=10)
al.pl.top_gene_changes(dr, n_top=10)

For a sample query, `signature_changes` uses the patient's own most specific genes: the 100 most above and 100 most below its cohort, capped for display.

In [ ]:
al.pl.signature_changes(dr, n_top=10, max_genes=30)

## The same data as an AnnData

In [ ]:
adata = al.datasets.beataml_demo()
adata

In [ ]:
# one observation; the other observations are the background
sid2 = samples.index[samples.group == "CBFB-MYH11"][0]
mt_cbf = al.tl.match(atlas, adata, sample=sid2, name=f"{sid2} CBFB-MYH11")
al.pl.pseudotime(mt_cbf, n_top=5)

**Pseudobulk per group** works the same way for single-cell data, e.g. `groupby="leiden"` / `group="3"`.

Here each genotype group is pooled, and the other two groups act as the background. With only three groups that background is small: for real single-cell data, use many clusters or samples.

In [ ]:
res = {}
for g in ["KMT2A-r", "NPM1-mut", "CBFB-MYH11"]:
    m = al.tl.match(atlas, adata, groupby="group", group=g, name=f"BeatAML {g} (pseudobulk)")
    res[g] = (m, al.tl.rank_drugs(atlas, m, cell_lines=5))
    print(g, "->", ", ".join(f"{c} [{t}]" for c, t in m.table.genotype.head(5).items()))

In [ ]:
al.pl.group_summary(res, title="BeatAML genotype groups (pseudobulk)")

In [ ]:
m, d = res["KMT2A-r"]
al.pl.summary(m, d, n_top=10)
al.pl.class_trends(d, classes=["menin-KMT2A", "DOT1L", "BET", "FLT3", "BCL2", "DNMT/HMA"])